In [ ]:
import os
import pickle
import time
import re
import pandas as pd
import numpy as np
import scipy.signal as sig
import warnings
from scipy.io import savemat
from scipy.signal import welch
from open_ephys.analysis import Session

In [2]:
%matplotlib osx
import matplotlib
matplotlib.use('macosx')
print("macOS native interactive windows enabled")
import matplotlib.pyplot as plt

macOS native interactive windows enabled


2026-06-07 22:43:36.798 Python[44610:3047596] WARNING: Secure coding is automatically enabled for restorable state! However, not on all supported macOS versions of this application. Opt-in to secure coding explicitly by implementing NSApplicationDelegate.applicationSupportsSecureRestorableState:.


### *CHANNEL SELECTION OF THE LFP DATA CAN BE PERFORMED USING THE PROCESSED VERSION OF THE DATA, AFTER WHICH, THE RAW DATA ONLY FROM THE SELECTED CHANNELS CAN BE PROCESSED AND DOWNSAMPLED USING THE UPDATED APPROACH. THIS WILL SAVE A LOT OF TIME AS THE DOWNSAMPLING SCRIPT HAS A VERY LONG RUNTIME.*

### **CHANGING FILE NAMES**

In [ ]:
# Define dataset
animal_id = 'r20'
condition = '2_fear conditioning'

def rename_files_in_folder(folder_path):
    """Rename files in the specified folder to remove prefixes and keep only the number."""
    # List all files in the specified folder
    for file_name in os.listdir(folder_path):
        # Use a regex to find the number in the filename
        match = re.match(rf'{animal_id}ch_(\d+)_ds\.pickle', file_name)
        if match:
            # Extract the number from the filename
            number = match.group(1)  # Extracts "0", "1", etc.
            # Construct the new file name
            new_file_name = f"{number}.pickle"  # "0.pickle", "1.pickle"
            # Create full paths for the old and new file names
            old_file_path = os.path.join(folder_path, file_name)
            new_file_path = os.path.join(folder_path, new_file_name)
            # Rename the file
            os.rename(old_file_path, new_file_path)
            print(f'Renamed: {file_name} -> {new_file_name}')

def main():
    # Specify the folder containing the files to rename
    folder_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/dataset/{animal_id}/{condition}'

    # Call the rename function
    rename_files_in_folder(folder_path)

if __name__ == "__main__":
    main()  # Execute the main function when the script is run


### **ASSIGNING TETRODE DATA - BUILDING DATAFRAME**

In [ ]:
'''This script creates a dataframe by assigning the tetrode to the channel maps. 
It includes the following components:
1. File: store pickle files as strings
2. Area: store brain regions
3. Tetrode: store the tetrode number
4. Leads: store the channel order (1st-4th) in the tetrode
This dataframe is saved in the file named assigned_dataframe '''

def load_channel_map(animal):
    file_path = os.path.join("/Volumes/TOSHIBA_UO/Joao_LFP/dataset/Channel maps", f"{animal}.pickle")
    with open(file_path, 'rb') as file:
        return pickle.load(file)

def load_condition_data(base_path, animal, condition):
    folder_path = os.path.join(base_path, f"dataset/{animal}/{condition}")
    condition_data = {}
    
    files = sorted(
        [f for f in os.listdir(folder_path) if f.endswith('.pickle')],
        key=lambda x: int(re.match(r'(\d+)\.pickle', x).group(1))
    )
    
    for file_name in files:
        index = int(re.match(r'(\d+)\.pickle', file_name).group(1))
        file_path = os.path.join(folder_path, file_name)
        
        try:
            with open(file_path, 'rb') as f:
                first_byte = f.read(1)
                if not first_byte:  # Empty file
                    print(f"{file_name}: SKIPPING - EMPTY (0 bytes)")
                    continue
                
                first_byte_ord = first_byte[0]
                if first_byte_ord != 0x80:
                    print(f"{file_name}: SKIPPING - INVALID (0x{first_byte_ord:02x})")
                    continue
                
                # Reset file pointer to beginning for pickle.load
                f.seek(0)
                condition_data[(animal, index)] = pickle.load(f)
                print(f"{file_name}: LOADED SUCCESSFULLY")
                
        except Exception as e:
            print(f"{file_name}: ERROR ({e})")
            continue
    
    return condition_data

def assign_condition_data(channel_map, condition_data, animal):
    assigned_data = []
    for index, channel in enumerate(channel_map):
        data = condition_data.get((animal, index))
        if data is not None:
            assigned_data.append({'Channel': channel, 'File': f"{index}.pickle"})
    return pd.DataFrame(assigned_data)

def transform_dataframe(assigned_df):
    assigned_df['Area'] = assigned_df['Channel'].str.split('TT').str[0].str.strip()
    assigned_df['Tetrode'] = assigned_df['Channel'].str.extract(r'(TT\d+)')[0]
    assigned_df['Leads'] = (assigned_df.index % 4) + 1
    return assigned_df[['File', 'Area', 'Tetrode', 'Leads']]

def save_assigned_data(animal, condition, transformed_data):
    save_folder = '/Volumes/TOSHIBA_UO/Joao_LFP/dataset/Assigned dataframes'
    os.makedirs(save_folder, exist_ok=True)
    save_path = os.path.join(save_folder, f"{animal}_{condition}_assigned.pickle")
    with open(save_path, 'wb') as file:
        pickle.dump(transformed_data, file)
    print(f"Saved assigned data for {animal} ({condition}) at: {save_path}")

def main():
    animals = ["r14"] # e.g. 'r16', 'r19', 'r20'
    conditions = ['2_fear conditioning'] #'fear_conditioning', 'probe_testing', 'extinction_training', 'extinction_testing'
    base_path = '/Volumes/TOSHIBA_UO/Joao_LFP'

    for animal in animals:
        channel_map = load_channel_map(animal)
        
        for condition in conditions:
            print(f"\n--- Processing {animal} - {condition} ---")
            condition_data = load_condition_data(base_path, animal, condition)
            assigned_data = assign_condition_data(channel_map, condition_data, animal)

            # Transform the DataFrame
            transformed_data = transform_dataframe(assigned_data)
            print(transformed_data)      
            save_assigned_data(animal, condition, transformed_data)

if __name__ == "__main__":
    main()



--- Processing r20 - 2_fear conditioning ---
0.pickle: LOADED SUCCESSFULLY
1.pickle: LOADED SUCCESSFULLY
2.pickle: LOADED SUCCESSFULLY
3.pickle: LOADED SUCCESSFULLY
4.pickle: LOADED SUCCESSFULLY
5.pickle: LOADED SUCCESSFULLY
6.pickle: LOADED SUCCESSFULLY
7.pickle: LOADED SUCCESSFULLY
8.pickle: LOADED SUCCESSFULLY
9.pickle: LOADED SUCCESSFULLY
10.pickle: LOADED SUCCESSFULLY
11.pickle: LOADED SUCCESSFULLY
12.pickle: LOADED SUCCESSFULLY
13.pickle: LOADED SUCCESSFULLY
14.pickle: LOADED SUCCESSFULLY
15.pickle: LOADED SUCCESSFULLY
16.pickle: LOADED SUCCESSFULLY
17.pickle: LOADED SUCCESSFULLY
18.pickle: LOADED SUCCESSFULLY
19.pickle: LOADED SUCCESSFULLY
20.pickle: LOADED SUCCESSFULLY
21.pickle: LOADED SUCCESSFULLY
22.pickle: LOADED SUCCESSFULLY
23.pickle: LOADED SUCCESSFULLY
24.pickle: LOADED SUCCESSFULLY
25.pickle: LOADED SUCCESSFULLY
26.pickle: LOADED SUCCESSFULLY
27.pickle: LOADED SUCCESSFULLY
28.pickle: LOADED SUCCESSFULLY
29.pickle: LOADED SUCCESSFULLY
30.pickle: LOADED SUCCESSFULLY
31.

### **LOAD SAVED DATAFRAME**
**RUN THIS EVERY TIME - SUBSEQUENT BLOCKS DEPEND ON THE DATAFRAME**

In [ ]:
# Define dataset
animal = "r16"
condition = "1_habituation"
path = "/Volumes/TOSHIBA_UO/Joao_LFP/dataset/Assigned dataframes"
file = f"{animal}_{condition}_assigned.pickle"

animal_condition_assigned = pd.read_pickle(f"{path}/{file}")

print(animal_condition_assigned.head())
print(animal_condition_assigned.info())
print(animal_condition_assigned.sample(5))

       File Area Tetrode  Leads Quality
0  0.pickle   A1     TT5      1     Bad
1  1.pickle   A1     TT5      2     Bad
2  2.pickle   A1     TT5      3    Good
3  3.pickle   A1     TT5      4     Bad
4  4.pickle   A1     TT2      1    Good
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   File     128 non-null    object
 1   Area     128 non-null    object
 2   Tetrode  128 non-null    object
 3   Leads    128 non-null    int64 
 4   Quality  128 non-null    object
dtypes: int64(1), object(4)
memory usage: 5.1+ KB
None
           File Area Tetrode  Leads Quality
13    13.pickle   A1     TT4      2     Bad
57    57.pickle  HPC    TT15      2    Good
4      4.pickle   A1     TT2      1    Good
112  112.pickle  BLA    TT23      1    Good
122  122.pickle  BLA    TT21      3     Bad


### **VISUALISING RAW LFP SIGNALS**

In [ ]:
# Visualising raw LFP signals of all channels from the same tetrode

class LFPZoomViewer:
    def __init__(self, base_path, brain_region, tetrode):
        self.base_path = base_path
        self.brain_region = brain_region
        self.tetrode = tetrode
        self.window_size = 16493978
        self.current_start = 0
        self.data_list = []
        
        # Performance settings
        plt.rcParams['path.simplify'] = True
        plt.rcParams['path.simplify_threshold'] = 0.5
        plt.rcParams['agg.path.chunksize'] = 10000
        
        self.colors = ['blue', 'green', 'orange', 'purple'] # one colur per tetrode channel
        self.fig, self.ax = plt.subplots(figsize=(16, 10))
        
        self.load_tetrode_channels() # load data for all channels on this tetrode
        self.update_plot()
        self.fig.canvas.mpl_connect('key_press_event', self.on_key_press)
        print("Controls: A=left, L=right, HOME=reset | Use mouse wheel/zoom toolbar for smooth zoom!")
    
    def load_tetrode_channels(self):
        # Matching the specified brain region and tetrode using the loaded dataframe
        chanels_from_specified_tetrode = animal_condition_assigned[
            (animal_condition_assigned['Area'] == self.brain_region) &
            (animal_condition_assigned['Tetrode'] == self.tetrode)]
        
        specified_channels_files = chanels_from_specified_tetrode['File'].tolist()
        
        # Load each channel's pickle file and store it
        for i, file_name in enumerate(specified_channels_files):
            file_path = os.path.join(self.base_path, file_name)
            data = self.load_data(file_path)
            if data is not None:
                file_index = int(file_name.replace('.pickle', ''))
                label = f'{self.brain_region} {self.tetrode} ch{file_index}'
                print(f"Loaded {len(data):,} samples: {label}")
                self.data_list.append((data, label, self.colors[i]))
    
    def load_data(self, file_path):
    # Load a single channel's data from disk
        try:
            with open(file_path, 'rb') as f:
                return pickle.load(f)
        except:
            return None
    
    def update_plot(self):
        self.ax.clear()
        start = max(0, self.current_start)
        end = min(start + self.window_size, len(self.data_list[0][0]) if self.data_list else 0)
        
        # Optimisation for speed
        # Plots fewer points when the visible window is wide so rendering stays fast and shows full resolution when zoomed in close
        num_points = end - start
        if num_points > 500000: # Very wide view
            subsample = max(50, num_points // 200000) # 200k points max
        elif num_points > 100000: # Medium view
            subsample = max(10, num_points // 50000)
        else: # Close-up
            subsample = 1
        
        # Plot each channel's segment for the current window
        for data, label, color in self.data_list:
            if end > start and start < len(data):
                segment = data[start:end]
                display_points = segment[::subsample]
                # Convert sample index to time in ms (0.02 ms/sample = 50 kHz sampling rate)
                time_ms = np.arange(len(display_points)) * 0.02 * subsample
                self.ax.plot(
                    time_ms, display_points, 
                    label=label, color=color, 
                    linewidth = 0.8, alpha = 0.8
                )
        
        # Axes
        self.ax.set_title(f'{self.brain_region} {self.tetrode} | FULL RECORDING ({end/50000:.0f}s) | Subsample:{subsample}', fontsize=16)
        self.ax.set_xlabel('Time (ms)')
        self.ax.set_ylabel('LFP amplitude (µV)')
        self.ax.legend(title='Channels', loc='upper right')
        self.ax.grid(True, alpha=0.3)
        plt.tight_layout()
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
    
    def on_key_press(self, event):
        step = 1000000
        if event.key == 'l': 
            self.current_start += step
        elif event.key == 'a': 
            self.current_start = max(0, self.current_start - step)
        elif event.key == 'home': 
            self.current_start = 0
        elif event.key == 'r':
            self.ax.set_xlim(0, self.window_size * 0.02)
            self.ax.set_ylim(np.nanmin([d[0][self.current_start:self.current_start+100000] for d in self.data_list if self.current_start+100000 < len(d[0])]), 
                           np.nanmax([d[0][self.current_start:self.current_start+100000] for d in self.data_list if self.current_start+100000 < len(d[0])]))
        self.update_plot()

# Define dataset
animal_id = 'r14'
condition = '2_fear conditioning'
base_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/dataset//{animal_id}/{condition}/'
viewer = LFPZoomViewer(base_path, 'PFC', 'TT4')
plt.show(block = True)


Loaded 18,593,272 samples: PFC TT2 ch4
Loaded 18,593,272 samples: PFC TT2 ch5
Loaded 18,593,272 samples: PFC TT2 ch6
Loaded 18,593,272 samples: PFC TT2 ch7
Controls: A=left, L=right, HOME=reset | Use mouse wheel/zoom toolbar for smooth zoom!


In [ ]:
# Visualising raw LFP signals of specified channels
# Specify channels to visualise

class LFPMultiChannelViewer:
    def __init__(self, base_path, channel_files, animal_condition_assigned):
        self.base_path = base_path
        self.channel_files = channel_files
        self.animal_condition_assigned = animal_condition_assigned
        self.window_size = 16_493_978
        self.current_start = 0
        self.data_list = []
        self.brain_region = None
        self.tetrode = None
        
        # Plot setup with performance optimisations
        plt.rcParams['path.simplify'] = True
        plt.rcParams['path.simplify_threshold'] = 0.5
        plt.rcParams['agg.path.chunksize'] = 10_000
        
        # Colours
        self.custom_colors = ['#1f77b4', '#d62728', '#ff7f0e', '#9467bd']
        self.tab10_colors = plt.cm.tab10(np.linspace(0, 1, 20))
        
        self.fig, self.ax = plt.subplots(figsize=(16, 10))
        
        # Load specified channels
        self.load_channels()
        
        # Initial plot and controls
        self.update_plot()
        self.fig.canvas.mpl_connect('key_press_event', self.on_key_press)
        print(f"Loaded {len(self.data_list)} channels:")
        for _, label, color in self.data_list:
            print(f"  - {label} (color: {color})")
        print("Controls: A=left, L=right, HOME=reset, R=reset zoom | Mouse scroll/zoom!")
    
    def load_channels(self):
        """Load specified channel files with metadata + detect common tetrode"""
        self.data_list = []
        tetrodes = set()
        regions = set()
        
        for i, channel_file in enumerate(self.channel_files):
            row = self.animal_condition_assigned[self.animal_condition_assigned['File'] == channel_file]
            if row.empty:
                print(f"Warning: No metadata for {channel_file}, skipping")
                continue
                
            brain_region = row['Area'].iloc[0]
            tetrode = row['Tetrode'].iloc[0]
            
            # Track for title
            regions.add(brain_region)
            tetrodes.add(tetrode)
            
            file_path = os.path.join(self.base_path, channel_file)
            data = self.load_data(file_path)
            
            if data is not None:
                channel_num = int(channel_file.replace('.pickle', ''))
                label = f'{brain_region} {tetrode} ch{channel_num}'
                
                # Assign colours
                if i < 4:
                    color = self.custom_colors[i]
                else:
                    color = self.tab10_colors[i % len(self.tab10_colors)]
                
                self.data_list.append((data, label, color))
                print(f"Loaded {len(data):,} samples: {label}")
        
        # Set titles
        if len(tetrodes) == 1 and len(regions) == 1:  # Same tetrode/region
            first_row = self.animal_condition_assigned[self.animal_condition_assigned['File'] == self.channel_files[0]]
            self.brain_region = first_row['Area'].iloc[0]
            self.tetrode = first_row['Tetrode'].iloc[0]
        else:
            self.brain_region = "Multi-region"
            self.tetrode = "Multi-tetrode"
    
    def load_data(self, file_path):
        try:
            with open(file_path, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            return None
    
    def update_plot(self):
        self.ax.clear()
        if not self.data_list:
            self.ax.text(0.5, 0.5, "No data loaded", ha='center', va='center', fontsize=16)
            self.fig.canvas.draw()
            return
        
        start = max(0, self.current_start)
        end = min(start + self.window_size, len(self.data_list[0][0]))
        num_points = end - start
        
        if num_points > 500_000:
            subsample = max(50, num_points // 200_000)
        elif num_points > 100_000:
            subsample = max(10, num_points // 50_000)
        else:
            subsample = 1
        
        for data, label, color in self.data_list:
            if end > start and start < len(data):
                segment = data[start:end]
                display_points = segment[::subsample]
                time_ms = np.arange(len(display_points)) * 0.02 * subsample
                
                self.ax.plot(time_ms, display_points, 
                           color=color, linewidth = 0.8, alpha = 0.8, 
                           label=label)
        
        if self.brain_region != "Multi-region":
            title = f'{self.brain_region} {self.tetrode} | Samples {start:,}–{end:,} | Subsample:{subsample}'
        else:
            title = f'Multi-region LFP | Samples {start:,}–{end:,} | Subsample:{subsample}'
        
        # Axes    
        self.ax.set_title(title, fontsize=16)
        self.ax.set_xlabel('Time (ms)')
        self.ax.set_ylabel('LFP amplitude (µV)')
        self.ax.legend(loc='upper right', fontsize=10)
        self.ax.grid(True, alpha=0.3)
        plt.tight_layout()
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
    
    def on_key_press(self, event):
        step = 1_000_000
        if event.key == 'l':
            self.current_start += step
        elif event.key == 'a':
            self.current_start = max(0, self.current_start - step)
        elif event.key == 'home':
            self.current_start = 0
        elif event.key == 'r':
            self.ax.set_xlim(0, self.window_size * 0.02)
            if self.data_list:
                visible_data = np.concatenate([
                    d[self.current_start:self.current_start+100_000] 
                    for d, _, _ in self.data_list 
                    if self.current_start+100_000 <= len(d)
                ])
                if len(visible_data) > 0:
                    self.ax.set_ylim(np.nanmin(visible_data), np.nanmax(visible_data))
        self.update_plot()

# Define dataset
animal_id = 'r16'
condition = '1_habituation'
base_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/dataset/{animal_id}/{condition}/'
specified_channels = ['65.pickle', '66.pickle'] # SPECIFY CHANNELS

viewer = LFPMultiChannelViewer(base_path, specified_channels, animal_condition_assigned)
plt.show(block = True)


✓ Loaded 17,756,306 samples: BLA TT31 ch65
Loaded 1 channels:
  - BLA TT31 ch65 (color: #1f77b4)
Controls: A=left, L=right, HOME=reset, R=reset zoom | Mouse scroll/zoom!


### **POWER SPECTRUM ANALYSIS**

In [ ]:
# Plotting PSD for all channels of a specified tetrode

# Specify dataset
animal_id = 'r14'
condition = '1_habituation'
brain_region = 'PFC'

# Specify tetrode
tetrode = 'TT8'

# Filter dataframe for specified brain region and tetrode
chanels_from_specified_tetrode = animal_condition_assigned[
    (animal_condition_assigned['Area'] == brain_region) &
    (animal_condition_assigned['Tetrode'] == tetrode)]

# Extract the four channel files and their quality
specified_channels_files = chanels_from_specified_tetrode['File'].tolist()

print("Files for", brain_region, tetrode, ":", specified_channels_files)

# Base path to the pickle files
base_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/dataset/{animal_id}/{condition}/'

# Sampling frequency
fs = 1000

# Colours
channel_colors = ['blue', 'green', 'orange', 'purple']

# Initialise the plot
plt.figure(figsize = (12, 8))

# Loop through each file, load the data, and plot.
for i, file_name in enumerate(specified_channels_files):
    # Add '.pickle' extension to the file name
    file_index = int(file_name.replace('.pickle', ''))
    # Construct the full file path
    file_path = os.path.join(base_path, file_name)

    # Load the LFP data from the pickle file
    with open(file_path, 'rb') as f:
        lfp_data = pickle.load(f)

    # Apply Welch's method to estimate the PSD
    frequencies, psd_values = welch(lfp_data, fs, nperseg = 1024)

    # Use specified channel colour
    color = channel_colors[i % len(channel_colors)]
    label = f'{brain_region} {tetrode} ch{file_index}'

    # Plot the PSD with a label corresponding to the file name and assign the channel colour
    plt.semilogy(frequencies, psd_values, label = label, color = color, linewidth = 1.5)

# Axes
plt.title(f"PSD: {brain_region} {tetrode}")
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD (V²/Hz)')

# Legend
plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')

# Layout
plt.grid(True, alpha = 0.3)
plt.tight_layout()
plt.show()


Files for PFC TT8 : ['20.pickle', '21.pickle', '22.pickle', '23.pickle']


In [ ]:
# Plotting PSD for specified channels

# Specify dataset
animal_id = 'r14'
condition = '2_fear conditioning'
brain_region = 'A1'

# Specify tetrode
tetrode = 'TT6'

# List of pickle file indices (specific channels) to plot
file_indices = [0, 1] # SPECIFY CHANNELS

# Filter dataframe to get qualities for the specified channel indices
chanels_from_specified_tetrode = animal_condition_assigned[
    (animal_condition_assigned['Area'] == brain_region) &
    (animal_condition_assigned['Tetrode'] == tetrode) &
    (animal_condition_assigned['File'].str.replace('.pickle', '').astype(int).isin(file_indices))
]

# Extract file names and qualities
specified_channels_files = chanels_from_specified_tetrode['File'].tolist()

print("Files for", brain_region, tetrode, ":", specified_channels_files)

# Base path to the pickle files
base_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/dataset/{animal_id}/{condition}/'

# Sampling frequency
fs = 1000

# Colours
colors = ['blue', 'green', 'orange', 'purple']

# Initialise the plot
plt.figure(figsize=(12, 8))

# Loop through each specified file, load the data, and plot.
for i, file_name in enumerate(specified_channels_files):
    file_index = int(file_name.replace('.pickle', ''))
    file_path = os.path.join(base_path, file_name)
    
    # Load the LFP data from the pickle file
    with open(file_path, 'rb') as f:
        lfp_data = pickle.load(f)
    
    # Apply Welch's method to estimate the PSD
    frequencies, psd_values = welch(lfp_data, fs, nperseg=1024)
    
    # Use specified channel colour
    color = colors[i % len(colors)]
    label = f'{brain_region} {tetrode} ch{file_index}'
    
    # Plot the PSD
    plt.semilogy(frequencies, psd_values, label = label, color = color, linewidth = 1.5)

# Axes
plt.title(f"PSD: {brain_region} {tetrode}")
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD (V²/Hz)')

# Legend
plt.legend(bbox_to_anchor = (1.05, 1), loc = 'upper left')

# Layout
plt.grid(True, alpha = 0.3)
plt.tight_layout()
plt.show()


Files for A1 TT5 : ['0.pickle', '1.pickle']


### **PLOTTING RAW LFPs PER BRAIN STATE**

In [ ]:
# Plotting example raw LFP traces
# Generates a single plot with an example trace for each brain state

warnings.filterwarnings("ignore", category=DeprecationWarning)

class LFPMultiChannelViewer:
    def __init__(self, base_path, channel_files, animal_condition_assigned, sleep_score_path, amplitude_threshold=10000):
        self.base_path = base_path
        self.channel_files = channel_files
        self.animal_condition_assigned = animal_condition_assigned
        self.sleep_score_path = sleep_score_path
        self.amplitude_threshold = amplitude_threshold
        
        self.window_size = 5000 # 5 seconds at 1 ms/sample
        self.current_start = 0
        self.data_list = []
        self.brain_region = None
        self.tetrode = None
        self.sleep_scores = self.load_sleep_scores()
        
        # Performance settings
        plt.rcParams['path.simplify'] = True
        plt.rcParams['path.simplify_threshold'] = 0.5
        plt.rcParams['agg.path.chunksize'] = 10_000
        
        # Colours
        self.brain_state_colors = {
            'AW': '#B8860B',   # dark yellow
            'QW': '#00AA00',   # green
            'NREM': '#1f77b4', # blue
            'REM': '#d62728'   # red
        }
        self.brain_states = ['AW', 'QW', 'NREM', 'REM']
        
        # Create four subplots (one for each brain state) stacked vertically
        self.fig, self.axes = plt.subplots(len(self.brain_states), 1, figsize=(16, 12), sharex=True)
        
        # Load specified channels
        self.load_channels()
        
        # Find first continuous window for each brain state
        self.state_starts = {}
        for state in self.brain_states:
            self.current_start = self.find_first_continuous_state_window(state)
            self.state_starts[state] = self.current_start
        
        # Initialise plot and controls
        self.update_plot()
        self.fig.canvas.mpl_connect('key_press_event', self.on_key_press)
        print(f"Loaded {len(self.data_list)} channels:")
        for _, label in self.data_list:
            print(f"  - {label}")
        print("Controls: A=left, L=right, HOME=reset, R=reset zoom | Mouse scroll/zoom")
    
    def load_channels(self):
        """Load specified channel files with metadata + detect common tetrode"""
        self.data_list = []
        tetrodes = set()
        regions = set()
        
        for i, channel_file in enumerate(self.channel_files):
            row = self.animal_condition_assigned[self.animal_condition_assigned['File'] == channel_file]
            if row.empty:
                print(f"Warning: No metadata for {channel_file}, skipping")
                continue
                
            brain_region = row['Area'].iloc[0]
            tetrode = row['Tetrode'].iloc[0]
            
            regions.add(brain_region)
            tetrodes.add(tetrode)
            
            file_path = os.path.join(self.base_path, channel_file)
            data = self.load_data(file_path)
            
            if data is not None:
                channel_num = int(channel_file.replace('.pickle', ''))
                label = f'{brain_region} {tetrode} ch{channel_num}'
                
                self.data_list.append((np.asarray(data), label))
                print(f"Loaded {len(data):,} samples: {label}")
        
        if len(tetrodes) == 1 and len(regions) == 1:
            first_row = self.animal_condition_assigned[self.animal_condition_assigned['File'] == self.channel_files[0]]
            self.brain_region = first_row['Area'].iloc[0]
            self.tetrode = first_row['Tetrode'].iloc[0]
        else:
            self.brain_region = "Multi-region"
            self.tetrode = "Multi-tetrode"
    
    def load_data(self, file_path):
    # Load a single LFP channel array
        try:
            with open(file_path, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading {file_path}: {e}")
            return None
    
    def load_sleep_scores(self):
        '''Load sleep score for a specified animal and condition, duplicating each score 5000 times.'''
        try:
            with open(self.sleep_score_path, 'rb') as file:
                data = pickle.load(file)
            
            if isinstance(data, list):
                array_data = np.asarray(data[-1])
            elif isinstance(data, np.ndarray):
                array_data = data
            else:
                array_data = np.atleast_1d(data)
            
            return np.array([float(score) for score in array_data for _ in range(1000)], dtype=float)
        except Exception as e:
            print(f"Error loading sleep scores: {e}")
            return np.array([])
    
    def find_first_continuous_state_window(self, brain_state):
        """Return the first start index where the next 5000 samples (5 seconds) are all the requested brain state."""
        state_scores = {'AW': 1.0, 'QW': 2.0, 'NREM': 3.0, 'REM': 4.0}
        target_score = state_scores[brain_state]
        
        if len(self.sleep_scores) < self.window_size:
            print(f"Warning: Sleep score vector is shorter than the requested 5-second window for {brain_state}.")
            return 0
        
        match = (self.sleep_scores == target_score)
        
        run = 0
        for i, ok in enumerate(match):
            if ok:
                run += 1
                if run >= self.window_size:
                    start = i - self.window_size + 1
                    print(f"Found continuous {brain_state} window at sample {start:,}")
                    return start
            else:
                run = 0
        
        print(f"Warning: No continuous 5-second {brain_state} segment found. Starting at 0.")
        return 0
    
    def update_plot(self):
        
        # Clear all axes
        for ax in self.axes:
            ax.clear()
        
        if not self.data_list:
            self.axes[0].text(0.5, 0.5, "No data loaded", ha='center', va='center', fontsize=16)
            self.fig.canvas.draw()
            return
        
        any_plotted = False
        
        # Plot each brain state in its own subplot
        for idx, state in enumerate(self.brain_states):
            ax = self.axes[idx]
            start = self.state_starts[state]
            end = min(start + self.window_size, len(self.data_list[0][0]))
            num_points = end - start
            
            if num_points <= 0:
                ax.text(0.5, 0.5, f"No {state} data", ha='center', va='center', fontsize=12)
                continue
            
            if num_points > 500_000:
                subsample = max(50, num_points // 200_000)
            elif num_points > 100_000:
                subsample = max(10, num_points // 50_000)
            else:
                subsample = 1
            
            state_plotted = False
            color = self.brain_state_colors[state]
            
            for data, label in self.data_list:
                if end > start and start < len(data):
                    segment = np.asarray(data[start:end], dtype=float)
                    
                    if np.nanmax(np.abs(segment)) > self.amplitude_threshold:
                        print(f"Skipping {label}: max abs amplitude exceeds {self.amplitude_threshold} for {state}")
                        continue
                    
                    display_points = segment[::subsample]
                    time_ms = np.arange(len(display_points)) * 1.0 * subsample
                    
                    ax.plot(time_ms, display_points,
                            color=color, linewidth=0.8, alpha=0.8)
                    state_plotted = True
                    any_plotted = True
            
            # Add subplot elements
            title = f'{state} LFP'
            ax.set_title(title, fontsize=20, fontweight='bold')
            ax.set_ylabel('μV', fontsize=18, fontweight='bold')
            ax.tick_params(axis='both', labelsize=16)
            ax.grid(True, alpha=0.3)
            ax.set_xlim(0, 5000)
            ax.set_ylim(-3000, 3000)
            ax.set_yticks([-3000, 0, 3000])
            
            if not state_plotted:
                ax.text(0.5, 0.5, f"No {state} segment passed threshold", ha='center', va='center', fontsize=12)
        
        # x-axis label only on the bottom plot
        self.axes[-1].set_xlabel('Time (ms)', fontsize=18, fontweight='bold')
        
        # Title
        if self.brain_region != "Multi-region":
            self.fig.suptitle(f'{self.brain_region} {self.tetrode} - All Brain States', fontsize=18, fontweight='bold', y=1.02)
        else:
            self.fig.suptitle('Multi-region LFP - All Brain States', fontsize=18, fontweight='bold', y=1.02)
        
        plt.tight_layout()
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
    
    def on_key_press(self, event):
        step = 250_000
        if event.key == 'l':
            self.current_start += step
            
            # Align all state starts
            for state in self.brain_states:
                self.state_starts[state] = max(0, self.current_start)
        elif event.key == 'a':
            self.current_start = max(0, self.current_start - step)
            for state in self.brain_states:
                self.state_starts[state] = self.current_start
        elif event.key == 'home':
            self.current_start = self.find_first_continuous_state_window('AW')
            for state in self.brain_states:
                self.state_starts[state] = self.current_start
        elif event.key == 'r':
            for ax in self.axes:
                ax.set_xlim(0, 5000)
            if self.data_list:
                visible_data = np.concatenate([
                    np.asarray(d[self.current_start:self.current_start + self.window_size], dtype=float)
                    for d, _ in self.data_list
                    if self.current_start + self.window_size <= len(d)
                ])
                if len(visible_data) > 0:
                    for ax in self.axes:
                        ax.set_ylim(np.nanmin(visible_data), np.nanmax(visible_data))
        self.update_plot()


# SPECIFY DATASET
animal_id = 'r16'
condition = '1_habituation'
base_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/dataset/{animal_id}/{condition}/'
sleep_score_path = f'/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/Sleep scores/{animal_id}_{condition}_sleep.pickle'
specified_channels = ['45.pickle']  # SPECIFY CHANNEL

viewer = LFPMultiChannelViewer(
    base_path,
    specified_channels,
    animal_condition_assigned,
    sleep_score_path,
    amplitude_threshold=4000
)
plt.show(block=True)


✓ Loaded 17,756,306 samples: HPC TT12 ch45
Found continuous AW window at sample 0
Found continuous QW window at sample 56,000
Found continuous NREM window at sample 44,000
Found continuous REM window at sample 384,000
Loaded 1 channels:
  - HPC TT12 ch45
Controls: A=left, L=right, HOME=reset, R=reset zoom | Mouse scroll/zoom!


### **LABELING TETRODE QUALITY**

In [ ]:
'''This script labels the quality of tetrode data and update the dataframe with a new column named Quality.'''

def load_assigned_data(animal, condition):
    """
    Load assigned tetrode data from a pickle file for the specified animal and condition.
    Returns A DataFrame containing the assigned data, or None if the file is not found.
    """
    file_path = os.path.join("/Volumes/TOSHIBA_UO/Joao_LFP/dataset/Assigned dataframes", f"{animal}_{condition}_assigned.pickle")
    
    try:
        with open(file_path, 'rb') as file:
            return pickle.load(file)
    except FileNotFoundError:
        print(f"Error: The file for {animal} under condition '{condition}' was not found.")
        return None

def label_quality(assigned_df, good_quality_indices):
    """
    Add a 'Quality' column to the DataFrame based on good quality indices.
    Returns the updated DataFrame with the 'Quality' column.
    """
    # Create a new column 'Quality' based on whether the index is in the good quality list
    assigned_df['Quality'] = assigned_df['File'].apply(
        lambda filename: 'Good' if int(filename.split('.')[0]) in good_quality_indices else 'Bad'
    )
    return assigned_df

def save_quality_data(animal, condition, assigned_df):
    """
    Save the updated DataFrame back to the original pickle file.
    """
    file_path = os.path.join("/Volumes/TOSHIBA_UO/Joao_LFP/dataset/Assigned dataframes", f"{animal}_{condition}_assigned.pickle")
    with open(file_path, 'wb') as file:
        pickle.dump(assigned_df, file)
    print(f"Updated data saved for {animal} under condition '{condition}'.")

def main():
    # Define the animal and good quality indices
    animal = "r20"
    condition = "2_fear conditioning"
    good_quality_indices = [106, 89, 92, 65, 71, 75, 77, 82, 
                            4, 9, 15, 2, 30, 17, 20, 87,
                            26, 43, 45, 49, 55, 57, 63, 35,
                            37, 116, 121, 125, 112, 109, 99, 103] # Specify which channels are to be labeled as "good"

    # Load the assigned data
    assigned_data = load_assigned_data(animal, condition)

    if assigned_data is not None:
        # Label the quality of the data
        labeled_data = label_quality(assigned_data, good_quality_indices)

        # Save the updated DataFrame back to the original file
        save_quality_data(animal, condition, labeled_data)

        # Display the updated DataFrame
        print(labeled_data)
    else:
        print("Data loading failed. Please check the file path and try again.")

if __name__ == "__main__":
    main()

Files for A1 TT6: []


/var/folders/bs/2dhpgpw17yl81kw5d059ptsw0000gq/T/ipykernel_86798/4179230648.py:92: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')


ZeroDivisionError: float division by zero

### **RELOAD UPDATED DATAFRAME**

In [ ]:
# IF NECESSARY

# Define dataset
animal = "r20"
condition = "2_fear conditioning"
path = "/Volumes/TOSHIBA_UO/Joao_LFP/dataset/Assigned dataframes"
file = f"{animal}_{condition}_assigned.pickle"

animal_condition_assigned = pd.read_pickle(f"{path}/{file}")

print(animal_condition_assigned.head())
print(animal_condition_assigned.info())
print(animal_condition_assigned.sample(5))

       File Area Tetrode  Leads Quality
0  0.pickle   A1     TT5      1     Bad
1  1.pickle   A1     TT5      2     Bad
2  2.pickle   A1     TT5      3    Good
3  3.pickle   A1     TT5      4     Bad
4  4.pickle   A1     TT2      1    Good
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   File     128 non-null    object
 1   Area     128 non-null    object
 2   Tetrode  128 non-null    object
 3   Leads    128 non-null    int64 
 4   Quality  128 non-null    object
dtypes: int64(1), object(4)
memory usage: 5.1+ KB
None
         File Area Tetrode  Leads Quality
12  12.pickle   A1     TT4      1     Bad
58  58.pickle  HPC    TT15      3     Bad
92  92.pickle  PFC    TT30      1    Good
83  83.pickle  PFC    TT35      4     Bad
67  67.pickle  PFC    TT31      4     Bad


## **PROCESSING RAW DATA AFTER CHANNEL SELECTION**

In [ ]:
# PROCESSING RAW LFP DATA - produces one pickle file per channel (FOR SELECTED CHANNELS)
# NO FILTERING OF 50 Hz - DFT and FieldTrip filter will be implemented later in the pipeline

# SPECIFY DATASET
animal = "r14"
condition = "1_habituation"

# Select which session to downsample
recording_dir = f"/Volumes/Joao Backup/raw dataset files/{animal}/{animal} {condition}/_{animal} {condition} raw data/continuous/Rhythm_FPGA-100.0"
data_path = os.path.join(recording_dir, "continuous.dat")

# Parameters
n_channels = 148 # total number of channels
Fs = 30000 # original sampling rate (Hz)
fs_target = 1000 # target sampling rate (Hz)
ds_factor = Fs // fs_target # downsampling factor

# Load the raw continuous data
print("Loading raw data...")
data = np.memmap(data_path, dtype="int16", mode = "r")
n_samples = data.size // n_channels
data = np.reshape(data, (n_samples, n_channels), order = "C")
print(f"Data loaded: {n_samples} samples × {n_channels} channels.")

# Channel selection - SPECIFY WHICH CHANNELS TO PROCESS
selected_channels = [31, 18, 21, 85, 24, 42, 47, 51, 53, 59, 61, 32, 37, 127, 112, 111, 98, 101, 105, 90, 94, 67, 70, 73, 78, 83]
output_dir = f"/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/{animal}/{animal} {condition}"
os.makedirs(output_dir, exist_ok = True)

# Processing only the selected channels
start_time = time.time()
print(f"Processing channels: {selected_channels}")

# Selecting and downsampling
for i in selected_channels:
    print(f"Processing channel {i}...")
    sig_tods = data[:, i]

    # Downsample
    downsampled_signal = sig.decimate(sig_tods, ds_factor)

    # Save each processed channel
    save_path = os.path.join(output_dir, f"{i}.pickle")
    with open(save_path, "wb") as handle:
        pickle.dump(downsampled_signal, handle, protocol = pickle.HIGHEST_PROTOCOL)

print(f"--- Completed in {time.time() - start_time:.2f} seconds ---")


## **MAKING EPOCHS**

In [ ]:
# MAKING EPOCHS FROM THE LFP DATA

warnings.filterwarnings("ignore", category = DeprecationWarning)

def load_specific_lfp_data(dataset_folder, specific_files):
    '''Load LFP data from specified pickle files.'''
    lfp_data_list = []
    
    for file_name in specific_files:
        file_path = os.path.join(dataset_folder, f'{file_name}.pickle') 
        print (file_path) 

        try:
            if os.path.exists(file_path):
                with open(file_path, 'rb') as file:
                    lfp_data = pickle.load(file)
                lfp_data_list.append(lfp_data)
            else:
                print(f'File not found: {file_path}')
        except Exception as e:
            print(f'Error loading {file_path}: {e}')
    
    return lfp_data_list

def load_sleep_scores(sleep_score_path):
    '''Load sleep score for a specified animal and condition, duplicating each score 5000 times.'''
    try:
        with open(sleep_score_path, 'rb') as file:
            data = pickle.load(file)
        
        # Handle both list and direct array formats
        if isinstance(data, list):
            array_data = np.asarray(data[-1])
        elif isinstance(data, np.ndarray):
            array_data = data
        else:
            array_data = np.atleast_1d(data)
        
        sleep_scores = [float(score) for score in array_data for _ in range(5000)]
        return sleep_scores
    except Exception as e:
        print(f"Error loading sleep scores: {e}")
        return []

def load_and_process_audio_timestamps(file_path):
    '''Load audio timestamps from a pickle file and convert them to integers.'''
    try:
        with open(file_path, 'rb') as file:
            timestamps = pickle.load(file)

        audio_timestamps = [int(ts) for sound_list in timestamps for ts in sound_list]
        return audio_timestamps
    except Exception as e:
        print(f"Error loading audio timestamps: {e}")
        return []

def mark_data_with_stimuli(lfp_data, audio_timestamps):
    '''Replace 3000 data points in LFP data with 'x' at each audio stimulus timestamp.'''
    marked_lfp = np.array(lfp_data, dtype=object)

    for stimulus_time in audio_timestamps:
        start_index = stimulus_time # Convert to zero-based index
        if start_index + 3000 <= len(marked_lfp):
            marked_lfp[start_index:start_index + 3000] = ['x'] * 3000
        else:
            print(f"Warning: Not enough data points to replace at index {start_index}.")
    return marked_lfp

def assign_data_with_sleepscore(marked_lfp, sleep_scores):
    """Segment LFP data based on sleep scores."""
    assigned_marked_lfp = {}
    num_sleep_scores = len(sleep_scores)
    
    for i in range(len(marked_lfp)):
        if i < num_sleep_scores:
            score = sleep_scores[i]
            if i not in assigned_marked_lfp:
                assigned_marked_lfp[i] = {'score': score, 'data': []}
            assigned_marked_lfp[i]['data'].append(marked_lfp[i])
        else:
            if i not in assigned_marked_lfp:
                assigned_marked_lfp[i] = {'score': None, 'data': []}
            assigned_marked_lfp[i]['data'].append(marked_lfp[i])

    return assigned_marked_lfp

def group_data(assigned_marked_lfp, target_state):
    """Group segments based on sleep scores and concatenate data."""
    
    # Map brain states to sleep scores:
    # AW = 1
    # QW = 2
    # NREM = 3
    # REM = 4
    state_scores = {'AW': 1.0, 'QW': 2.0, 'NREM': 3.0, 'REM': 4.0}
    target_score = state_scores[target_state]
    
    groups = {
        target_state: {'count': 0, 'data': []},
        'Unidentified': {'count': 0, 'data': []},  
    }
    
    unassigned_count = 0

    for index, items in assigned_marked_lfp.items():
        score = items['score']
        if score == target_score:
            groups[target_state]['count'] += 1
            groups[target_state]['data'].extend(items['data'])
        else:
            groups['Unidentified']['count'] += 1
            groups['Unidentified']['data'].extend(items['data'])

    for group in groups:
        groups[group]['total_data_points'] = len(groups[group]['data'])

    return groups, unassigned_count

def remove_x_values(groups):
    """Remove data points equal to 'x' from each brain state group and retain original indices."""
    for group in groups:
        original_indices = [i for i, value in enumerate(groups[group]['data']) if value != 'x']
        groups[group]['data'] = [value for value in groups[group]['data'] if value != 'x']
        groups[group]['total_data_points'] = len(groups[group]['data'])
        groups[group]['original_indices'] = original_indices # Store original indices
    return groups

def create_and_save_epochs(lfp_data_list, audio_timestamps, sleep_scores, save_folder, animal_id, condition, brain_state):
    """Create and save multiple epochs of REM data, each lasting 2 minutes (120,000 data points)."""
    epoch_length = 240000 # 4 minutes of data points
    epoch_count = 0 # Counter for the number of epochs created

    all_epochs_per_channel = [] # List to store all short epochs from each good channel

    # Process data for each specified channel
    for idx, lfp_data in enumerate(lfp_data_list):
        print(f'Processing data from channel with index {idx}:')

        marked_lfp = mark_data_with_stimuli(lfp_data, audio_timestamps)  
        assigned_marked_lfp = assign_data_with_sleepscore(marked_lfp, sleep_scores)
        brain_state_groups, unassigned_count = group_data(assigned_marked_lfp, brain_state)
        brain_state_groups = remove_x_values(brain_state_groups)

        target_data = brain_state_groups[brain_state]['data']
        original_indices = brain_state_groups[brain_state]['original_indices']

        # Create epochs for this channel
        consecutive_indices = []
        channel_epochs = [] # Store the epochs for this channel

        for i in range(len(target_data)):
            if target_data[i] != 'x':
                consecutive_indices.append(original_indices[i])
                # Check if we have enough consecutive data points
                if len(consecutive_indices) == epoch_length:
                    epoch_data = target_data[i - epoch_length + 1:i + 1] # Last 120000 data points
                    channel_epochs.append(epoch_data) # Add the epoch to the list for this channel
                    epoch_count += 1
                    consecutive_indices = [] # Reset for the next epoch
            else:
                consecutive_indices = [] # Reset if 'x' is encountered

        all_epochs_per_channel.append(channel_epochs) # Store epochs for this channel

    # Check if all channels have the same number of epochs
    num_epochs_per_channel = len(all_epochs_per_channel[0]) # Number of epochs in the first channel
    for epochs in all_epochs_per_channel:
        if len(epochs) != num_epochs_per_channel:
            print("Warning: Not all channels have the same number of epochs.")
            return

    # Create the matrices where each matrix contains the corresponding epochs from all channels
    for epoch_index in range(num_epochs_per_channel):
        matrix_data = []

        for channel_epochs in all_epochs_per_channel:
            matrix_data.append(channel_epochs[epoch_index]) # Add the same epoch from each channel

        # Save the matrix
        savemat(os.path.join(save_folder, f'{animal_id}_{condition}_matrix_{epoch_index}.mat'), {'matrix': matrix_data})
        print(f"Saved matrix {epoch_index} with corresponding epochs from all channels")

    print(f"Total epochs saved: {epoch_count}")

# SPECIFY DATASET
def main():
    base_path = '/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset'
    animal_id = 'r14'
    condition = '2_fear conditioning'
    area = 'A1' # Specify target brain region
    brain_state = 'AW' # Specify target brain state

    timestamps_path = os.path.join(base_path, 'Audio timestamps', f'{animal_id}_{condition}_sleep.pickle')
    sleep_score_path = os.path.join(base_path, 'Sleep scores', f'{animal_id}_{condition}_sleep.pickle')
    dataset_folder = os.path.join(base_path, animal_id, f'{animal_id} {condition}')
    save_folder = os.path.join(base_path, 'Saved matrices', f'{animal_id} matrices', f'{animal_id} {condition} matrices', area, f'{area} {brain_state}', f'4-minute epochs ({area} {brain_state})')
    os.makedirs(save_folder, exist_ok = True) # Create folder if it doesn't exist

    specific_files = ['24', '42', '47', '51', '53', '59', '61', '32'] # Specify the pickle files (channels) to be processed
    
    lfp_data_list = load_specific_lfp_data(dataset_folder, specific_files)

    if lfp_data_list:
        sleep_scores = load_sleep_scores(sleep_score_path)
        audio_timestamps = load_and_process_audio_timestamps(timestamps_path)

        # Call the function to create and save epochs
        create_and_save_epochs(lfp_data_list, audio_timestamps, sleep_scores, save_folder, animal_id, condition, brain_state)

    else:
        print('No LFP data to process.')
        return  # Exit if no LFP data

if __name__ == '__main__':
    main()


/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/24.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/42.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/47.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/51.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/53.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/59.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/61.pickle
/Volumes/TOSHIBA_UO/Joao_LFP/raw dataset/r14/r14 1_habituation/32.pickle
Processing data from channel with index 0:
Processing data from channel with index 1:
Processing data from channel with index 2:
Processing data from channel with index 3:
Processing data from channel with index 4:
Processing data from channel with index 5:
Processing data from channel with index 6:
Processing data from channel with index 7:
Saved matrix 0 with corresponding epochs from all channels
Saved matrix 

END